In [ ]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pyarrow pandas scipy accelerate tabulate
import laya, transformers, datasets, torch
print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("PyTorch version     :", torch.__version__)

In [ ]:
WORK_DIR = "/tmp/laya_snowflake"
os.makedirs(WORK_DIR, exist_ok=True)
TRAIN_SCRIPT = os.path.join(WORK_DIR, "train_rlcd.py")

In [ ]:
import json
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Snowpark returns OBJECT/VARIANT columns as JSON strings
ds_train = (
    session.table("JEV_ZENDESK_LAB_V2.CURATED.TYPED_DECISIONS")
    .filter("SPLIT = 'train'")
    .select("STATE", "QUESTIONS", "GOLD")
    .collect()
)

rows = []
for row in ds_train:
    rows.append({
        "state": json.loads(row["STATE"]),
        "questions": json.loads(row["QUESTIONS"]),
        "gold": json.loads(row["GOLD"]),
    })

print(len(rows))
rows[0]

In [ ]:
import os, json, torch
from datasets import load_dataset
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.train import items_from_rows

MODEL_ID = "convaiinnovations/laya"
print(f"Fetching tokenizer and config from {MODEL_ID}...")
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)


items, skipped = items_from_rows(tok, rows, cfg["max_len"], cfg["head_max_len"])
print(f"Preprocessed {len(items)} training items (skipped {skipped}) across 1,200 cases.")
torch.save(items, os.path.join(WORK_DIR, "train_items.pt"))
print("Saved preprocessed items to /kaggle/working/train_items.pt")


In [ ]:
%%writefile {TRAIN_SCRIPT}
"""2xT4 DDP training script for the typed-decisions fine-tune.

Training semantics -- item construction, encoding/collation, the RLCD objective,
the sigma schedule, the calibration holdout and the temperature fit -- come from
the canonical primitives in `laya.train` / `laya.common` / `laya.calibrate`.
What intentionally stays notebook-private is the distributed orchestration:
process-group setup, rank/local-rank wiring, the DDP wrapper, equal-rank
sharding and the effective-batch accounting.
"""
import os, sys, time, json, random, math
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file
from transformers import AutoTokenizer

from laya.common import build_model, collate_items
from laya.calibrate import fit_temperature_map
from laya.train import (
    calibration_records,
    encode_item,
    rlcd_loss,
    save_checkpoint,
    sigma_at,
    split_calibration,
)

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]
    work_dir = sys.argv[3]

    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 1024
    cfg["head_max_len"] = 256
    max_len = cfg["max_len"]
    head_max_len = cfg["head_max_len"]

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))

    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)

    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)

    all_items = torch.load(os.path.join(work_dir, "train_items.pt"), weights_only=False)

    # Held out before sharding: `split_calibration` is seed-deterministic and
    # rank-independent, so every rank withholds the same items, and the shard
    # split happens after -- the calibration slice never reaches a training batch.
    train_items, calib_items = split_calibration(all_items, calib_max=400, calib_frac=0.1, seed=20260922)

    # Trim before sharding so every rank gets the same count (drops at most
    # world_size-1 items). Otherwise ranks can run different numbers of
    # micro-batches and DDP deadlocks on the unmatched all-reduce (#678).
    train_items = train_items[:len(train_items) // world_size * world_size]
    my_items = train_items[rank::world_size]

    EPOCHS = 4
    MICRO_BATCH = 8      # 8 sequences per forward pass per GPU
    GRAD_ACCUM = 4       # Effective batch across 2 GPUs = 64 sequences (8 * 2 * 4)
    GROUP_SIZE = 4       # RLCD baseline samples
    LR_ENCODER = 2.5e-5  # Encoder adaptation rate
    LR_HEAD = 1.0e-4     # Head adaptation rate
    SIGMA_START = 0.4    # Exploration noise
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]

    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEAD}
    ], weight_decay=0.01)

    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    if rank == 0:
        print(f"Starting 2xT4 DDP training: {len(train_items)} train items ({len(calib_items)} held out for calibration) | {len(my_items)} per rank | {EPOCHS} epochs")
    t0 = time.time()

    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0

        sigma = sigma_at(epoch, EPOCHS, SIGMA_START, SIGMA_END)
        steps_per_epoch = math.ceil(len(my_items) / MICRO_BATCH)

        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue

            encoded = [encode_item(tok, it, max_len, head_max_len) for it in chunk]
            batch = collate_items([encoded], tok.pad_token_id)

            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device)
                )

            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            target = batch["target"].to(device)

            # Canonical objective: RLCD policy gradient over noisy logits + full
            # soft cross-entropy guidance (`laya.train.rlcd_loss`).
            loss = rlcd_loss(logits, target, mask, batch["qtype"].to(device), sigma,
                             samples=GROUP_SIZE, w_sph=0.75, w_rps=1.0)

            window_start = (accum_step // GRAD_ACCUM) * GRAD_ACCUM
            window_size = min(GRAD_ACCUM, steps_per_epoch - window_start)
            scaled = loss / window_size
            scaler.scale(scaled).backward()
            accum_step += 1

            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            epoch_loss += loss.item()
            n_batches += 1

            if rank == 0 and (n_batches % 50) == 0:
                cur_lr = scheduler.get_last_lr()[0]
                print(f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches} | Loss: {loss.item():.4f} | LR: {cur_lr:.2e}")

        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} Completed in {time.time()-t0:.1f}s | Avg Loss: {epoch_loss/max(1, n_batches):.4f} ===")

        dist.barrier()

        # Overwrite a single rolling checkpoint after each epoch so a crash,
        # OOM, or Kaggle session timeout doesn't lose all prior training.
        if rank == 0:
            ckpt_dir = os.path.join(output_dir, "checkpoint_latest")
            save_checkpoint(model, tok, dict(cfg, max_len=max_len, head_max_len=head_max_len), ckpt_dir)
            with open(os.path.join(ckpt_dir, "checkpoint_meta.json"), "w") as f:
                json.dump({
                    "epoch": epoch + 1,
                    "total_epochs": EPOCHS,
                    "avg_loss": epoch_loss / max(1, n_batches)
                }, f, indent=2)
            print(f"  Saved rolling checkpoint (epoch {epoch+1}/{EPOCHS}) to {ckpt_dir}")

    dist.barrier()

    # Post-training temperature calibration on rank 0
    if rank == 0:
        print("\nFitting post-training calibration temperatures...")
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache()
        model.eval()
        try:
            records = calibration_records(model, tok, calib_items, device, max_len, head_max_len, batch_size=16)
            fitted = fit_temperature_map(records)
            fitted_temps = fitted["temperature"]
            print("Fitted calibration temperatures (choice, score, noul):", [round(t, 3) for t in fitted_temps])
        except Exception as e:
            print("Temperature fitting fallback:", e)
            fitted_temps, fitted = None, {"temperature_by_options": {}}

        out_cfg = dict(cfg, max_len=max_len, head_max_len=head_max_len)
        out_cfg["fine_tuned"] = True
        out_cfg["model_name"] = "laya-typed-decisions"
        if fitted_temps is not None:
            out_cfg["temperature"] = fitted_temps
            # This fit carries per-bucket overrides; inherited ones would shadow them.
            out_cfg.pop("temperature_by_options", None)
            if fitted["temperature_by_options"]:
                out_cfg["temperature_by_options"] = fitted["temperature_by_options"]
        save_checkpoint(model, tok, out_cfg, output_dir)
        print(f"Model successfully saved to {output_dir}!")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()

In [ ]:
import sys, torch

OUTPUT_DIR = os.path.join(WORK_DIR, "laya_finetuned")
MODEL_DIR = model_dir

n_gpus = torch.cuda.device_count()
print("GPUs visible:", n_gpus)
assert n_gpus > 0, "No GPU available on this service"

cmd = f"{sys.executable} -m torch.distributed.run --standalone --nproc_per_node={n_gpus} {TRAIN_SCRIPT} {MODEL_DIR} {OUTPUT_DIR} {WORK_DIR}"
print("Executing DDP training:", cmd)
!{cmd}

In [ ]:
import time, json
import numpy as np
import pandas as pd
from datasets import load_dataset
import laya
from laya.common import ece_score

print("Loading test split for evaluation...")
ds_test = (
    session.table("JEV_ZENDESK_LAB_V2.CURATED.TYPED_DECISIONS")
    .filter("SPLIT = 'test'")
    .select("ID", "WORKFLOW", "STATE", "QUESTIONS", "GOLD")
    .collect()
)

rows = []
for row in ds_test:
    rows.append({
        "id": row["ID"],
        "workflow": row["WORKFLOW"],
        "state": json.loads(row["STATE"]),
        "questions": json.loads(row["QUESTIONS"]),
        "gold": json.loads(row["GOLD"]),
    })

agent_ft = laya.Agent(OUTPUT_DIR, device="cuda")

predictions = []
latencies_ms = []

print(f"Evaluating {len(rows)} test cases on GPU...")
t0_eval = time.time()

for i, row in enumerate(rows):
    case_id = row["id"]
    workflow = row["workflow"]
    # already parsed into dicts above
    state = row["state"]
    questions = row["questions"]
    gold = row["gold"]
    
    t0 = time.perf_counter()
    res = agent_ft.predict(state, questions)
    dt_ms = (time.perf_counter() - t0) * 1000
    latencies_ms.append(dt_ms)
    
    predictions.append({
        "id": case_id,
        "workflow": workflow,
        "pred": res["answers"],
        "gold": gold,
        "questions": questions,
        "latency_ms": dt_ms
    })

print(f"Evaluated all {len(ds_test)} cases in {time.time() - t0_eval:.1f}s!")

In [ ]:
accuracies = []
soft_accuracies = []
brier_scores = []
kl_divs = []
tv_distances = []
score_maes = []
within_one = []
all_confs = []
all_corrects = []

for item in predictions:
    pred_answers = item["pred"]
    gold_answers = item["gold"]
    questions = item["questions"]
    
    for qid, qdef in questions.items():
        p_ans = pred_answers[qid]
        g_ans = gold_answers[qid]
        q_type = qdef["type"]
        
        # 1. Choice
        if q_type == "choice":
            keys = list(qdef["criteria"].keys())
            pred_choice = p_ans["choice"]
            gold_label = str(g_ans["label"])
            
            is_corr = float(pred_choice == gold_label)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)
            
            p_probs = np.array([p_ans["probabilities"].get(k, 1e-6) for k in keys])
            g_probs = np.array([g_ans["probabilities"].get(k, 1e-6) for k in keys])
            p_probs /= p_probs.sum()
            g_probs /= g_probs.sum()
            
            all_confs.append(float(p_probs.max()))
            soft_accuracies.append(float((p_probs * g_probs).sum()))
            brier_scores.append(float(((p_probs - g_probs) ** 2).sum()))
            tv_distances.append(float(0.5 * np.abs(p_probs - g_probs).sum()))
            kl_divs.append(float((g_probs * np.log(np.clip(g_probs / p_probs, 1e-12, 1e4))).sum()))
            
        # 2. Noul
        elif q_type == "noul":
            p_val = p_ans["noul"]
            g_val = g_ans.get("noul", g_ans.get("probabilities", {}).get("true", 0.5))
            gold_label = str(g_ans["label"]).lower()
            
            pred_label = "true" if p_val >= 0.5 else "false"
            is_corr = float(pred_label == gold_label)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)
            all_confs.append(float(max(p_val, 1.0 - p_val)))
            
            p_dist = np.array([1.0 - p_val, p_val])
            g_dist = np.array([1.0 - g_val, g_val])
            
            soft_accuracies.append(float((p_dist * g_dist).sum()))
            brier_scores.append(float(((p_dist - g_dist) ** 2).sum()))
            tv_distances.append(float(0.5 * np.abs(p_dist - g_dist).sum()))
            kl_divs.append(float((g_dist * np.log(np.clip(g_dist / p_dist, 1e-12, 1e4))).sum()))
            
        # 3. Score
        elif q_type == "score":
            p_score = p_ans["score"]
            g_score = g_ans.get("score", 0.0)
            score_maes.append(abs(p_score - g_score))
            within_one.append(float(abs(p_score - g_score) <= 1.0))
            
            # Use argmax of probabilities for discrete prediction
            n_levels = len(qdef.get("criteria", []))
            p_probs_score = np.array([p_ans["probabilities"].get(str(i), 0.0) for i in range(n_levels)])
            if p_probs_score.sum() > 0:
                p_probs_score /= p_probs_score.sum()
                p_lvl = int(np.argmax(p_probs_score))
                all_confs.append(float(p_probs_score.max()))
            else:
                p_lvl = int(round(p_score))
                all_confs.append(0.5)
                
            g_lvl = int(g_ans.get("label", int(round(g_score))))
            is_corr = float(p_lvl == g_lvl)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)

laya_acc = np.mean(accuracies)
laya_soft_acc = np.mean(soft_accuracies)
laya_brier = np.mean(brier_scores)
laya_kl = np.mean(kl_divs)
laya_tv = np.mean(tv_distances)
laya_ece = ece_score(np.array(all_confs), np.array(all_corrects))
laya_mae = np.mean(score_maes) if score_maes else 0.0
laya_within1 = np.mean(within_one) if within_one else 0.0
laya_latency = np.percentile(latencies_ms, 50)

comparison_data = [
    {
        "Model": "TypeSafe Jev 1.13.0 Banking77",
        "Kind": "general",
        "Accuracy": 0.727,
        "Soft Acc": 0.580,
        "Brier": 0.148,
        "ECE": 0.144,
        "Score MAE": 0.391,
        "Within 1 Level": "0.952",
        "ms/case": 710
    },
    {
        "Model": "Laya Fine Tuned Enterprise Data",
        "Kind": "fine-tuned",
        "Accuracy": round(laya_acc, 3),
        "Soft Acc": round(laya_soft_acc, 3),
        "Brier": round(laya_brier, 3),
        "ECE": round(laya_ece, 3),
        "Score MAE": round(laya_mae, 3),
        "Within 1 Level": f"{laya_within1:.3f}",
        "ms/case": round(laya_latency, 1),
        "Cost/Case": "$0.00 (Self-Hosted)"
    }
]

df_comp = pd.DataFrame(comparison_data)
print("=== HEAD-TO-HEAD BENCHMARK TABLE ===\n")
print(df_comp.to_markdown(index=False))

In [ ]:
import json
import pandas as pd
import torch
import transformers
import laya
from snowflake.ml.model import custom_model
from snowflake.ml.registry import Registry

REG_DB, REG_SCHEMA = "JEV_ZENDESK_LAB_V2", "CURATED"
# V4: CUDA runtime + torch as a version range. An exact "torch==X" pin gets rewritten by the registry to the
# installed local build ("2.13.0+cu129"), which the image build cannot install (V2/V3 failed on this).
MODEL_NAME, VERSION = "LAYA_ZENDESK_TRIAGE", "V4"


def _object_frame(data: dict) -> pd.DataFrame:
    # pandas 3 defaults to a string-dtype column index, which snowflake-ml's signature checks reject
    df = pd.DataFrame({k: pd.Series(v, dtype=object) for k, v in data.items()})
    df.columns = pd.Index(list(data.keys()), dtype=object)
    return df


class LayaTriageModel(custom_model.CustomModel):
    """Wraps the fine-tuned laya.Agent. Input: STATE, QUESTIONS (JSON strings). Output: ANSWERS (JSON string)."""

    def __init__(self, context: custom_model.ModelContext) -> None:
        super().__init__(context)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.agent = laya.Agent(self.context["model_dir"], device=device)

    @custom_model.inference_api
    def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
        answers = []
        for state, questions in zip(input_df["STATE"], input_df["QUESTIONS"]):
            res = self.agent.predict(json.loads(state), json.loads(questions))
            answers.append(json.dumps(res["answers"], default=str))
        return _object_frame({"ANSWERS": answers})


laya_model = LayaTriageModel(custom_model.ModelContext(artifacts={"model_dir": OUTPUT_DIR}))

# Sample input in the same shape users will pass (JSON text from the curated table)
sample_input = _object_frame({
    "STATE": [json.dumps(r["state"]) for r in rows[:3]],
    "QUESTIONS": [json.dumps(r["questions"]) for r in rows[:3]],
})
print("Local test:\n", laya_model.predict(sample_input).head())

torch_major_minor = ".".join(torch.__version__.split("+")[0].split(".")[:2])

reg = Registry(session=session, database_name=REG_DB, schema_name=REG_SCHEMA)
mv = reg.log_model(
    laya_model,
    model_name=MODEL_NAME,
    version_name=VERSION,
    sample_input_data=sample_input,
    pip_requirements=[
        f"laya=={laya.__version__}",
        f"torch>={torch_major_minor},<{torch_major_minor}.99",
        f"transformers=={transformers.__version__}",
        "safetensors",
    ],
    target_platforms=["SNOWPARK_CONTAINER_SERVICES"],
    options={"cuda_version": torch.version.cuda},
    comment="Laya fine-tuned on JEV_ZENDESK_LAB_V2.CURATED.TYPED_DECISIONS (train split) for Zendesk triage",
)
print(f"Registered {REG_DB}.{REG_SCHEMA}.{mv.model_name} version {mv.version_name}")

# Confirm the saved requirements carry no local "+cuXXX" tag before building an image from them
import tempfile
_tmp = tempfile.mkdtemp()
session.file.get(f"snow://model/{REG_DB}.{REG_SCHEMA}.{MODEL_NAME}/versions/{VERSION}/model/env/requirements.txt", _tmp)
saved_reqs = open(f"{_tmp}/requirements.txt").read()
print("Saved requirements:\n" + saved_reqs)
assert "+cu" not in saved_reqs, "Requirements still pin a local CUDA build tag"
mv.show_functions()

In [ ]:
import time, json
import numpy as np
import pandas as pd
from snowflake.ml.registry import Registry
from snowflake.snowpark import functions as F

SERVICE_NAME = "LAYA_ZENDESK_TRIAGE_SVC"
COMPUTE_POOL = "JEV_LAB_GPU_POOL"

reg = Registry(session=session, database_name="JEV_ZENDESK_LAB_V2", schema_name="CURATED")
# V4: logged with a CUDA runtime and PyPI-resolvable requirements, so it can be served on the GPU pool
mv_reg = reg.get_model("LAYA_ZENDESK_TRIAGE").version("V4")

# Model is SPCS-only, so predictions run through an inference service (created once, reused after)
existing = mv_reg.list_services()
if existing.empty or SERVICE_NAME not in existing["name"].str.split(".").str[-1].values:
    mv_reg.create_service(
        service_name=SERVICE_NAME,
        service_compute_pool=COMPUTE_POOL,
        gpu_requests="1",
        max_instances=1,
        ingress_enabled=False,
    )

# Same test split as before, with STATE/QUESTIONS serialized to JSON text (the model's input signature)
test_df = (
    session.table("JEV_ZENDESK_LAB_V2.CURATED.TYPED_DECISIONS")
    .filter("SPLIT = 'test'")
    .select(
        "ID", "WORKFLOW",
        F.to_json(F.col("STATE")).alias("STATE"),
        F.to_json(F.col("QUESTIONS")).alias("QUESTIONS"),
        F.to_json(F.col("GOLD")).alias("GOLD"),
    )
)
n_cases = test_df.count()

print(f"Scoring {n_cases} test cases via {SERVICE_NAME}...")
t0_eval = time.time()
svc_out = mv_reg.run(test_df, function_name="predict", service_name=SERVICE_NAME).to_pandas()
elapsed = time.time() - t0_eval

# One batched call: per-case latency is the amortized average, not per-request latency
per_case_ms = elapsed * 1000 / max(n_cases, 1)

predictions_svc = []
for _, r in svc_out.iterrows():
    predictions_svc.append({
        "id": r["ID"],
        "workflow": r["WORKFLOW"],
        "pred": json.loads(r["ANSWERS"]),
        "gold": json.loads(r["GOLD"]),
        "questions": json.loads(r["QUESTIONS"]),
        "latency_ms": per_case_ms,
    })
latencies_ms_svc = [p["latency_ms"] for p in predictions_svc]

print(f"Scored {len(predictions_svc)} cases in {elapsed:.1f}s ({per_case_ms:.1f} ms/case amortized)")

In [ ]:
accuracies = []
soft_accuracies = []
brier_scores = []
kl_divs = []
tv_distances = []
score_maes = []
within_one = []
all_confs = []
all_corrects = []

for item in predictions_svc:
    pred_answers = item["pred"]
    gold_answers = item["gold"]
    questions = item["questions"]
    
    for qid, qdef in questions.items():
        p_ans = pred_answers[qid]
        g_ans = gold_answers[qid]
        q_type = qdef["type"]
        
        # 1. Choice
        if q_type == "choice":
            keys = list(qdef["criteria"].keys())
            pred_choice = p_ans["choice"]
            gold_label = str(g_ans["label"])
            
            is_corr = float(pred_choice == gold_label)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)
            
            p_probs = np.array([p_ans["probabilities"].get(k, 1e-6) for k in keys])
            g_probs = np.array([g_ans["probabilities"].get(k, 1e-6) for k in keys])
            p_probs /= p_probs.sum()
            g_probs /= g_probs.sum()
            
            all_confs.append(float(p_probs.max()))
            soft_accuracies.append(float((p_probs * g_probs).sum()))
            brier_scores.append(float(((p_probs - g_probs) ** 2).sum()))
            tv_distances.append(float(0.5 * np.abs(p_probs - g_probs).sum()))
            kl_divs.append(float((g_probs * np.log(np.clip(g_probs / p_probs, 1e-12, 1e4))).sum()))
            
        # 2. Noul
        elif q_type == "noul":
            p_val = p_ans["noul"]
            g_val = g_ans.get("noul", g_ans.get("probabilities", {}).get("true", 0.5))
            gold_label = str(g_ans["label"]).lower()
            
            pred_label = "true" if p_val >= 0.5 else "false"
            is_corr = float(pred_label == gold_label)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)
            all_confs.append(float(max(p_val, 1.0 - p_val)))
            
            p_dist = np.array([1.0 - p_val, p_val])
            g_dist = np.array([1.0 - g_val, g_val])
            
            soft_accuracies.append(float((p_dist * g_dist).sum()))
            brier_scores.append(float(((p_dist - g_dist) ** 2).sum()))
            tv_distances.append(float(0.5 * np.abs(p_dist - g_dist).sum()))
            kl_divs.append(float((g_dist * np.log(np.clip(g_dist / p_dist, 1e-12, 1e4))).sum()))
            
        # 3. Score
        elif q_type == "score":
            p_score = p_ans["score"]
            g_score = g_ans.get("score", 0.0)
            score_maes.append(abs(p_score - g_score))
            within_one.append(float(abs(p_score - g_score) <= 1.0))
            
            # Use argmax of probabilities for discrete prediction
            n_levels = len(qdef.get("criteria", []))
            p_probs_score = np.array([p_ans["probabilities"].get(str(i), 0.0) for i in range(n_levels)])
            if p_probs_score.sum() > 0:
                p_probs_score /= p_probs_score.sum()
                p_lvl = int(np.argmax(p_probs_score))
                all_confs.append(float(p_probs_score.max()))
            else:
                p_lvl = int(round(p_score))
                all_confs.append(0.5)
                
            g_lvl = int(g_ans.get("label", int(round(g_score))))
            is_corr = float(p_lvl == g_lvl)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)

svc_acc = np.mean(accuracies)
svc_soft_acc = np.mean(soft_accuracies)
svc_brier = np.mean(brier_scores)
svc_kl = np.mean(kl_divs)
svc_tv = np.mean(tv_distances)
svc_ece = ece_score(np.array(all_confs), np.array(all_corrects))
svc_mae = np.mean(score_maes) if score_maes else 0.0
svc_within1 = np.mean(within_one) if within_one else 0.0
svc_latency = np.percentile(latencies_ms_svc, 50)

comparison_data_svc = [
    {
        "Model": "TypeSafe Jev 1.13.0 Banking77",
        "Kind": "general",
        "Accuracy": 0.727,
        "Soft Acc": 0.580,
        "Brier": 0.148,
        "ECE": 0.144,
        "Score MAE": 0.391,
        "Within 1 Level": "0.952",
        "ms/case": 710
    },
    {
        "Model": "Laya Fine Tuned Enterprise Data",
        "Kind": "fine-tuned",
        "Accuracy": round(laya_acc, 3),
        "Soft Acc": round(laya_soft_acc, 3),
        "Brier": round(laya_brier, 3),
        "ECE": round(laya_ece, 3),
        "Score MAE": round(laya_mae, 3),
        "Within 1 Level": f"{laya_within1:.3f}",
        "ms/case": round(laya_latency, 1),
        "Cost/Case": "$0.00 (Self-Hosted)"
    }
]

df_comp_svc = pd.DataFrame(comparison_data_svc)
print("=== HEAD-TO-HEAD BENCHMARK TABLE (Registry Service) ===\n")
print(df_comp_svc.to_markdown(index=False))